# Извлечение признаков FEI Face: FaceNet encoder + Decoder (v2)

**Проблема v1**: Ratio inter/intra = 2.02, PSNR = 17 dB — недостаточно для NCT.

**Улучшения v2**:
1. **L2-нормализация** эмбеддингов → признаки на единичной сфере, стабильные расстояния
2. **Center Loss** — явно минимизирует расстояние от центра класса (λ=0.5)
3. **Аугментация** — flip, color jitter, rotation → инвариантность внутри класса
4. **Увеличен вес классификации** (λ_cls=2.0) и добавлен cosine annealing

Цель: Ratio inter/intra > 5, PSNR > 20 dB

In [ ]:
!pip install facenet-pytorch --no-deps -q

In [ ]:
import os, glob, re
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from facenet_pytorch import InceptionResnetV1
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
import kagglehub
path = kagglehub.dataset_download("whizzkid/fei-face-data")
print("Path to dataset files:", path)

In [ ]:
# --- Константы ---
FEI_ROOT = path
IMG_SIZE = 160
LATENT_DIM = 512
NUM_CLASSES = 200
BATCH_SIZE = 32
NUM_EPOCHS = 80
LR = 1e-3

LAMBDA_PERC = 0.5
LAMBDA_SHARP = 1.0
LAMBDA_CLS = 1.0
LAMBDA_CENTER = 0.3

In [ ]:
class FEIFaceDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        parts = [os.path.join(root, f"originalimages_part{i}") for i in range(1, 5)]
        self.files = []
        self.labels = []
        for p in parts:
            if os.path.isdir(p):
                for fpath in sorted(glob.glob(os.path.join(p, "*.jpg"))):
                    fname = os.path.basename(fpath)
                    match = re.match(r"(\d+)-(\d+)\.jpg", fname)
                    if match:
                        pid = int(match.group(1))
                        if 1 <= pid <= 200:
                            self.files.append(fpath)
                            self.labels.append(pid - 1)
        print(f"Загружено {len(self.files)} изображений, {len(set(self.labels))} лиц")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


transform = transforms.Compose([
    transforms.CenterCrop(480),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

dataset = FEIFaceDataset(root=FEI_ROOT, transform=transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
eval_loader = DataLoader(dataset, batch_size=256, shuffle=False, num_workers=2)

## Архитектура: FaceNet encoder + Classification head + Decoder

In [ ]:
class FaceNetEncoder(nn.Module):
    """InceptionResnetV1 (VGGFace2) → L2-нормализованный 512-мерный вектор"""
    def __init__(self, latent_dim=512, freeze_backbone=True):
        super().__init__()
        self.backbone = InceptionResnetV1(pretrained='vggface2')
        
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
        
        self.projector = nn.Sequential(
            nn.Linear(512, latent_dim),
            nn.BatchNorm1d(latent_dim),
        )
        self.scale = nn.Parameter(torch.tensor(16.0))
    
    def forward(self, x):
        h = self.backbone(x)
        z = self.projector(h)
        z = F.normalize(z, p=2, dim=1) * self.scale
        return z


class CenterLoss(nn.Module):
    def __init__(self, num_classes, feat_dim):
        super().__init__()
        self.centers = nn.Parameter(torch.randn(num_classes, feat_dim))
    
    def forward(self, z, labels):
        return F.mse_loss(z, self.centers[labels])


class ConvDecoder(nn.Module):
    """Улучшенный декодер: 512 → 160x160 RGB (double-conv + LeakyReLU)"""
    def __init__(self, latent_dim=512):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 512 * 5 * 5)
        
        self.decoder = nn.Sequential(
            # 5→10
            nn.ConvTranspose2d(512, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),
            nn.Conv2d(256, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),
            
            # 10→20
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            
            # 20→40
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            
            # 40→80
            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),
            nn.Conv2d(32, 32, 3, 1, 1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),
            
            # 80→160
            nn.ConvTranspose2d(32, 16, 4, 2, 1),
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.2),
            nn.Conv2d(16, 3, 3, 1, 1),
            nn.Sigmoid(),
        )
    
    def forward(self, z):
        h = self.fc(z)
        h = h.view(-1, 512, 5, 5)
        return self.decoder(h)


class SharpnessLoss(nn.Module):
    """L1 + Gradient loss — сохраняет границы и текстуры"""
    def forward(self, recon, target):
        l1 = F.l1_loss(recon, target)
        dx_r = recon[:,:,:,1:] - recon[:,:,:,:-1]
        dx_t = target[:,:,:,1:] - target[:,:,:,:-1]
        dy_r = recon[:,:,1:,:] - recon[:,:,:-1,:]
        dy_t = target[:,:,1:,:] - target[:,:,:-1,:]
        grad = F.l1_loss(dx_r, dx_t) + F.l1_loss(dy_r, dy_t)
        return l1 + grad


class FaceAutoencoder(nn.Module):
    def __init__(self, latent_dim=512, num_classes=200, freeze_backbone=True):
        super().__init__()
        self.encoder = FaceNetEncoder(latent_dim, freeze_backbone)
        self.decoder = ConvDecoder(latent_dim)
        self.classifier = nn.Linear(latent_dim, num_classes)
        self.center_loss = CenterLoss(num_classes, latent_dim)
    
    def forward(self, x):
        z = self.encoder(x)
        recon = self.decoder(z)
        logits = self.classifier(z)
        return recon, z, logits
    
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, z):
        return self.decoder(z)

## Perceptual Loss

In [ ]:
class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features
        self.block1 = nn.Sequential(*list(vgg)[:4]).eval()
        self.block2 = nn.Sequential(*list(vgg)[:9]).eval()
        self.block3 = nn.Sequential(*list(vgg)[:16]).eval()
        for param in self.parameters():
            param.requires_grad = False
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    
    def normalize(self, x):
        return (x - self.mean) / self.std
    
    def forward(self, recon, target):
        r = self.normalize(recon)
        t = self.normalize(target)
        return (F.mse_loss(self.block1(r), self.block1(t)) +
                F.mse_loss(self.block2(r), self.block2(t)) +
                F.mse_loss(self.block3(r), self.block3(t)))

## Обучение (v3)

Loss = MSE + 0.5·Perceptual + 1.0·Sharpness + 1.0·CE + 0.3·CenterLoss

- **Perceptual** (VGG-16) — текстурная схожесть
- **Sharpness** (L1 + gradient) — чёткие границы
- **Gradient clipping** — стабильность обучения

In [ ]:
model = FaceAutoencoder(latent_dim=LATENT_DIM, num_classes=NUM_CLASSES, freeze_backbone=True).to(device)
perc_loss_fn = PerceptualLoss().to(device)
sharp_loss_fn = SharpnessLoss()

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(trainable_params, lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-5)

total_params = sum(p.numel() for p in trainable_params)
print(f'Обучаемых параметров: {total_params:,}')

In [ ]:
model.train()
for epoch in range(NUM_EPOCHS):
    ep_mse, ep_perc, ep_sharp, ep_cls, ep_ctr, ep_total = 0, 0, 0, 0, 0, 0
    correct, total = 0, 0
    
    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        
        recon, z, logits = model(imgs)
        
        mse = F.mse_loss(recon, imgs)
        perc = perc_loss_fn(recon, imgs)
        sharp = sharp_loss_fn(recon, imgs)
        cls = F.cross_entropy(logits, labels)
        ctr = model.center_loss(z, labels)
        
        loss = mse + LAMBDA_PERC * perc + LAMBDA_SHARP * sharp + LAMBDA_CLS * cls + LAMBDA_CENTER * ctr
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        ep_mse += mse.item()
        ep_perc += perc.item()
        ep_sharp += sharp.item()
        ep_cls += cls.item()
        ep_ctr += ctr.item()
        ep_total += loss.item()
        
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    
    scheduler.step()
    n = len(loader)
    acc = 100 * correct / total
    
    if epoch % 5 == 0 or epoch == NUM_EPOCHS - 1:
        print(f'Epoch [{epoch+1}/{NUM_EPOCHS}] '
              f'Loss: {ep_total/n:.4f} '
              f'(MSE: {ep_mse/n:.4f}, Perc: {ep_perc/n:.4f}, '
              f'Sharp: {ep_sharp/n:.4f}, CLS: {ep_cls/n:.4f}, CTR: {ep_ctr/n:.4f}) '
              f'Acc: {acc:.1f}%')

torch.save({
    'model_state_dict': model.state_dict(),
    'latent_dim': LATENT_DIM,
    'num_classes': NUM_CLASSES,
    'img_size': IMG_SIZE,
}, 'checkpoint_facenet_ae.pth')
print('Модель сохранена: checkpoint_facenet_ae.pth')

## Визуализация реконструкций

In [ ]:
model.eval()
with torch.no_grad():
    imgs, labels = next(iter(loader))
    imgs = imgs[:8].to(device)
    recon, z, _ = model(imgs)

fig, axes = plt.subplots(2, 8, figsize=(20, 5))
for i in range(8):
    axes[0, i].imshow(imgs[i].cpu().permute(1, 2, 0))
    axes[0, i].set_title('Original')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(recon[i].cpu().permute(1, 2, 0).clamp(0, 1))
    axes[1, i].set_title('Reconstructed')
    axes[1, i].axis('off')

plt.suptitle('FaceNet Encoder + Decoder (Perceptual + Classification Loss)', fontsize=14)
plt.tight_layout()
plt.show()

## Fine-tuning: разморозка части FaceNet

In [ ]:
FT_EPOCHS = 40

backbone = model.encoder.backbone
for name, param in backbone.named_parameters():
    if 'block8' in name or 'last' in name or 'repeat_3' in name:
        param.requires_grad = True

optimizer_ft = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-5)
scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=FT_EPOCHS, eta_min=1e-6)

ft_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Fine-tuning параметров: {ft_params:,}')

model.train()
for epoch in range(FT_EPOCHS):
    ep_loss, ep_ctr = 0, 0
    correct, total = 0, 0
    
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer_ft.zero_grad()
        
        recon, z, logits = model(imgs)
        mse = F.mse_loss(recon, imgs)
        perc = perc_loss_fn(recon, imgs)
        sharp = sharp_loss_fn(recon, imgs)
        cls = F.cross_entropy(logits, labels)
        ctr = model.center_loss(z, labels)
        loss = mse + LAMBDA_PERC * perc + LAMBDA_SHARP * sharp + LAMBDA_CLS * cls + LAMBDA_CENTER * ctr
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer_ft.step()
        ep_loss += loss.item()
        ep_ctr += ctr.item()
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    
    scheduler_ft.step()
    n = len(loader)
    if epoch % 5 == 0 or epoch == FT_EPOCHS - 1:
        print(f'FT [{epoch+1}/{FT_EPOCHS}] Loss: {ep_loss/n:.4f}, '
              f'CTR: {ep_ctr/n:.4f}, Acc: {100*correct/total:.1f}%')

torch.save({
    'model_state_dict': model.state_dict(),
    'latent_dim': LATENT_DIM,
    'num_classes': NUM_CLASSES,
    'img_size': IMG_SIZE,
}, 'checkpoint_facenet_ae_finetuned.pth')
print('Fine-tuned модель сохранена')

In [ ]:
# Визуализация после fine-tuning
model.eval()
with torch.no_grad():
    imgs, labels = next(iter(loader))
    imgs = imgs[:8].to(device)
    recon, z, _ = model(imgs)

fig, axes = plt.subplots(2, 8, figsize=(20, 5))
for i in range(8):
    axes[0, i].imshow(imgs[i].cpu().permute(1, 2, 0))
    axes[0, i].set_title('Original')
    axes[0, i].axis('off')
    axes[1, i].imshow(recon[i].cpu().permute(1, 2, 0).clamp(0, 1))
    axes[1, i].set_title('Reconstructed')
    axes[1, i].axis('off')

plt.suptitle('После Fine-tuning', fontsize=14)
plt.tight_layout()
plt.show()

## Анализ внутриклассовой дисперсии

In [ ]:
model.eval()

latents, labels_all = [], []
with torch.no_grad():
    for imgs, lbls in eval_loader:
        z = model.encode(imgs.to(device))
        latents.append(z.cpu().numpy())
        labels_all.append(lbls.numpy())

X_latent = np.vstack(latents)
y_labels = np.hstack(labels_all)
print(f'Латентное пространство: {X_latent.shape}')

intra_dists = []
for cls in range(NUM_CLASSES):
    X_cls = X_latent[y_labels == cls]
    center = X_cls.mean(axis=0)
    dists = np.linalg.norm(X_cls - center, axis=1)
    intra_dists.extend(dists)

centers = np.array([X_latent[y_labels == c].mean(axis=0) for c in range(NUM_CLASSES)])
inter_dists = []
for i in range(NUM_CLASSES):
    for j in range(i+1, NUM_CLASSES):
        inter_dists.append(np.linalg.norm(centers[i] - centers[j]))

intra_mean = np.mean(intra_dists)
inter_mean = np.mean(inter_dists)
ratio = inter_mean / intra_mean

print(f'\nВнутриклассовое расстояние (среднее): {intra_mean:.4f}')
print(f'Межклассовое расстояние (среднее): {inter_mean:.4f}')
print(f'Ratio (inter/intra): {ratio:.2f}')
print(f'(Чем больше ratio, тем лучше для NCT. Нужно > 3-5)')

# Cosine similarity внутри класса
from sklearn.metrics.pairwise import cosine_similarity
intra_cos = []
for cls in range(NUM_CLASSES):
    X_cls = X_latent[y_labels == cls]
    sim = cosine_similarity(X_cls)
    n = sim.shape[0]
    triu = sim[np.triu_indices(n, k=1)]
    intra_cos.extend(triu)
print(f'\nCosine similarity (внутриклассовая): {np.mean(intra_cos):.4f} ± {np.std(intra_cos):.4f}')
print(f'(Для NCT нужно > 0.9)')

## Извлечение признаков для NCT

In [ ]:
# Признаки извлекаются без аугментации (eval_loader)
X_ordered = []
for class_id in range(200):
    indices = np.where(y_labels == class_id)[0]
    X_class = X_latent[indices]
    assert X_class.shape == (14, 512), f"Класс {class_id}: {X_class.shape}"
    X_ordered.append(X_class)

X_3d = np.stack(X_ordered)  # (200, 14, 512)
print(f'Итоговый массив: {X_3d.shape}')

np.save('facenet_ae_classes_3d_fei_512.npy', X_3d)

X_2d = X_3d.reshape(-1, 512)
np.savetxt('facenet_ae_3d_data.csv', X_2d, delimiter=',')
print(f'CSV: {X_2d.shape}')

from IPython.display import FileLink
display(FileLink('facenet_ae_classes_3d_fei_512.npy'))
display(FileLink('facenet_ae_3d_data.csv'))
display(FileLink('checkpoint_facenet_ae_finetuned.pth'))

In [ ]:
# t-SNE
from sklearn.manifold import TSNE
import seaborn as sns

n_sample = min(2000, len(X_latent))
idx = np.random.choice(len(X_latent), n_sample, replace=False)

tsne = TSNE(n_components=2, perplexity=30, n_iter=500, random_state=42)
Z_tsne = tsne.fit_transform(X_latent[idx])

plt.figure(figsize=(10, 8))
sns.scatterplot(x=Z_tsne[:, 0], y=Z_tsne[:, 1], hue=y_labels[idx], palette='tab20', s=50, legend=False)
plt.title('t-SNE: FaceNet Autoencoder (FEI Face)')
plt.axis('off')
plt.show()

In [ ]:
model.eval()
total_mse, total_psnr, n = 0, 0, 0

with torch.no_grad():
    for imgs, _ in eval_loader:
        imgs = imgs.to(device)
        recon, _, _ = model(imgs)
        mse = F.mse_loss(recon, imgs, reduction='none').mean(dim=[1,2,3])
        psnr = -10 * torch.log10(mse + 1e-8)
        total_mse += mse.sum().item()
        total_psnr += psnr.sum().item()
        n += imgs.size(0)

print(f'Средний MSE: {total_mse/n:.6f}')
print(f'Средний PSNR: {total_psnr/n:.2f} dB')
print(f'(PSNR > 25 dB — хорошо, > 30 dB — отлично)')